In [3]:
import math
import skimage
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display

from save_dicom import save_as_dicom

# loading the image
try:
    img = skimage.io.imread("tomograf-obrazy/SADDLE_PE.JPG")
    img_grey = skimage.color.rgb2gray(img)
    img_grey = (img_grey * 255).astype(np.uint8)
    print(f"Loaded image shape: {img_grey.shape}")
except Exception as e:
    print(f"Error loading image: {e}")

def detectorCords(a, n, phi, r, i):
    if i == 0:
        xd = r * math.cos(a + math.pi - phi/2)
        yd = r * math.sin(a + math.pi - phi/2)
    elif i == n-1:
        xd = r * math.cos(a + math.pi + phi/2)
        yd = r * math.sin(a + math.pi + phi/2)
    else:
        xd = r * math.cos(a + math.pi - phi/2 + i * phi/(n-1))
        yd = r * math.sin(a + math.pi - phi/2 + i * phi/(n-1))
    return xd, yd

def get_pixels_on_line(x1, y1, x2, y2):
    start_y, start_x = int(round(y1)), int(round(x1))
    end_y, end_x = int(round(y2)), int(round(x2))
    rows, cols = skimage.draw.line_nd((start_y, start_x), (end_y, end_x), endpoint=True, integer=True)
    return rows, cols

def run_simulation(delta_a, n, phi_deg, name, patient_id, comments):
    curr_h, curr_w = img_grey.shape
    r = math.sqrt(curr_w**2 + curr_h**2) / 2 + 10 
    phi = math.radians(phi_deg)
    
    # lista wszystkich kątow detektorow
    angles = np.arange(0, 360, delta_a)
    num_steps = len(angles)
    # liczba skokow kąta x liczba detektorów
    sinogram = np.zeros((num_steps, n))
    offset_x, offset_y = curr_w / 2, curr_h / 2

    # generacja sinogramu
    for view, current_angle in enumerate(angles):
        current_angle_rad = math.radians(current_angle)
        # xe, ye - pozycja emitera
        xe = r * math.cos(current_angle_rad)
        ye = r * math.sin(current_angle_rad)
        for D in range(n):
            xd, yd = detectorCords(current_angle_rad, n, phi, r, D)
            rows, cols = get_pixels_on_line(xe + offset_x, ye + offset_y, 
                                            xd + offset_x, yd + offset_y)
            mask = (rows >= 0) & (rows < curr_h) & (cols >= 0) & (cols < curr_w)
            pixels = img_grey[rows[mask], cols[mask]]
            if len(pixels) > 0:
                sinogram[view, D] = np.mean(pixels) # średnia z jasności

    # rekonstrukcja obrazu z sinogramu

    # tu bedzie obraz wynikowy
    reconstruction = np.zeros((curr_h, curr_w))
    # tu beda liczniki do normalizacji
    hits = np.zeros((curr_h, curr_w))

    # iterujemy przez kazdy widok w sinogramie (kazdy kąt)
    for view, current_angle in enumerate(angles):
        current_angle_rad = math.radians(current_angle)
        # xe, ye - pozycja emitera
        xe = r * math.cos(current_angle_rad)
        ye = r * math.sin(current_angle_rad)

        # iterujemy przez kazdy detektor w widoku
        for D in range(n):
            # zczytanie wartosci zapisanej w sinogramie
            val = sinogram[view, D]

            # xe, ye - pozycja detektora
            xd, yd = detectorCords(current_angle_rad, n, phi, r, D)


            rows, cols = get_pixels_on_line(xe + offset_x, ye + offset_y, 
                                            xd + offset_x, yd + offset_y)
            mask = (rows >= 0) & (rows < curr_h) & (cols >= 0) & (cols < curr_w)
            reconstruction[rows[mask], cols[mask]] += val
            hits[rows[mask], cols[mask]] += 1

    reconstruction = np.divide(reconstruction, hits, out=np.zeros_like(reconstruction), where=hits!=0)
    
    # wizualizacja
    v_min, v_max = np.percentile(reconstruction, (5, 99))
    reconstructed_plot = skimage.exposure.rescale_intensity(reconstruction, in_range=(v_min, v_max))

    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(img_grey, cmap='gray')
    ax[0].set_title("Original Image")
    ax[1].imshow(sinogram, cmap='gray', aspect='auto')
    ax[1].set_title(f"Sinogram\n({num_steps} views)")
    ax[2].imshow(reconstructed_plot, cmap='gray')
    ax[2].set_title("Reconstruction")
    plt.show()

    # zapis do pliku dicom
    v_min_dcm, v_max_dcm = np.percentile(reconstruction, (0, 100))
    reconstructed_dcm = skimage.exposure.rescale_intensity(reconstruction, in_range=(v_min_dcm, v_max_dcm))

    patient_data = {
        "PatientName": name,
        "PatientID": patient_id,
        "ImageComments": comments
    }
    
    save_as_dicom("out.dcm", reconstructed_dcm, patient_data)

# gui
style = {'description_width': '120px'}

interact_manual(
    run_simulation,
    # parametry symulacji

    # rozmiar kroku
    delta_a = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Step (Δα):', style=style),
    # rozpietosc detektorow
    n = widgets.IntSlider(value=180, min=10, max=720, step=10, description='Detectors (n):', style=style),
    # rozpietosc ukladu w stopniach
    phi_deg = widgets.IntSlider(value=180, min=10, max=270, step=5, description='Spread (φ°):', style=style),
    
    # metadane 
    name = widgets.Text(value='Imie', placeholder='Enter Name', description='Patient Name:', style=style),
    patient_id = widgets.Text(value='123123', placeholder='Enter ID', description='Patient ID:', style=style),
    comments = widgets.Textarea(value='komentarz', placeholder='Comments', description='Comments:', style=style)
);

Loaded image shape: (876, 1053)


interactive(children=(FloatSlider(value=2.0, description='Step (Δα):', max=10.0, min=0.5, step=0.5, style=Slid…